In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


In [4]:
check = train[
    ["battle_turn", "round", "turn", "previous_hp", "pikachu_hp"]
].copy()

check["next_previous_hp"] = check["previous_hp"].shift(-1)

check["hp_matches"] = (
    check["pikachu_hp"] == check["next_previous_hp"]
)

print(check.head(20))
print("\nMatch percentage:",
      check["hp_matches"].mean() * 100)

             battle_turn  round  turn  previous_hp  pikachu_hp  \
0   Round 1 - Turn 00:00      1     0         54.0          54   
1   Round 1 - Turn 01:00      1     1         31.0          11   
2   Round 1 - Turn 02:00      1     2          7.0          12   
3   Round 1 - Turn 03:00      1     3         54.0           0   
4   Round 1 - Turn 04:00      1     4          NaN          29   
5   Round 1 - Turn 05:00      1     5         34.0          15   
6   Round 1 - Turn 06:00      1     6         54.0          54   
7   Round 1 - Turn 07:00      1     7         54.0          47   
8   Round 1 - Turn 08:00      1     8         18.0          31   
9   Round 1 - Turn 09:00      1     9         54.0          54   
10  Round 1 - Turn 10:00      1    10         54.0          54   
11  Round 1 - Turn 11:00      1    11         54.0          11   
12  Round 1 - Turn 12:00      1    12         17.0           2   
13  Round 1 - Turn 13:00      1    13         12.0           0   
14  Round 

In [5]:
print(train[[
    "battle_turn",
    "round",
    "turn",
    "opponent_pokemon",
    "previous_hp",
    "pikachu_hp"
]].head(30).to_string(index=False))

         battle_turn  round  turn opponent_pokemon  previous_hp  pikachu_hp
Round 1 - Turn 00:00      1     0        Charizard         54.0          54
Round 1 - Turn 01:00      1     1        Dragonite         31.0          11
Round 1 - Turn 02:00      1     2          Steelix          7.0          12
Round 1 - Turn 03:00      1     3        Charizard         54.0           0
Round 1 - Turn 04:00      1     4         Alakazam          NaN          29
Round 1 - Turn 05:00      1     5        Charizard         34.0          15
Round 1 - Turn 06:00      1     6         Venusaur         54.0          54
Round 1 - Turn 07:00      1     7        Blastoise         54.0          47
Round 1 - Turn 08:00      1     8         Alakazam         18.0          31
Round 1 - Turn 09:00      1     9        Dragonite         54.0          54
Round 1 - Turn 10:00      1    10         Vaporeon         54.0          54
Round 1 - Turn 11:00      1    11          Machamp         54.0          11
Round 1 - Tu

In [6]:
print(
    train.groupby("round")["turn"]
    .agg(["min", "max", "count"])
    .head(20)
)

       min  max  count
round                 
1        0   23     24
2        0   23     24
3        0   23     24
4        0   23     24
5        0   23     24
6        0   23     24
7        0   23     24
8        0   23     24
9        0   23     24
10       0   23     24
11       0   23     24
12       0   23     24
13       0   23     24
14       0   23     24
15       0   23     24
16       0   23     24
17       0   23     24
18       0   23     24
19       0   23     24
20       0   23     24


In [77]:
print(
    train[train["round"]==1][
        ["round","turn","opponent_level","pikachu_level","move_power","attack_stat","defense_stat","sp_attack_stat","sp_defense_stat","speed_stat_pikachu","speed_stat_opponent","attack_stage","defense_stage","speed_stage",
        "opponent_type","max_hp","previous_hp", "pikachu_hp"]
    ].to_string(index=False)
)

 round  turn  opponent_level  pikachu_level  move_power  attack_stat  defense_stat  sp_attack_stat  sp_defense_stat  speed_stat_pikachu  speed_stat_opponent  attack_stage  defense_stage  speed_stage opponent_type  max_hp  previous_hp  pikachu_hp
     1     0            27.0           26.0        90.0          NaN          37.0            29.0             43.0                48.0                 29.0           0.0            0.0          0.0          Fire    54.0         54.0          54
     1     1            32.0           26.0        80.0         34.0          26.0            39.0             49.0                55.0                 25.0           0.0           -1.0          0.0        Dragon    54.0         31.0          11
     1     2            24.0           26.0        40.0         30.0          31.0            33.0             38.0                46.0                 47.0          -1.0            1.0          0.0         Steel    54.0          7.0          12
     1     3    

In [60]:
print(train["held_item"].unique())

<StringArray>
['Sitrus Berry', <NA>, 'Assault Vest', 'Light Ball', 'Focus Sash']
Length: 5, dtype: string


In [72]:
print(train["pikachu_ability"].unique())

<StringArray>
['Static', 'Lightning Rod', <NA>]
Length: 3, dtype: string


In [68]:
print(train["weather_condition"].unique())

<StringArray>
['Clear', 'Rain', 'Sandstorm', 'Electric Terrain', 'Sun', <NA>, 'Hail']
Length: 7, dtype: string


In [ ]:
X = train.drop(columns=["pikachu_hp", "battle_turn"])
X_test = test.drop(columns=["battle_turn"])
target = "pikachu_hp"

# Clean column names
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

# Clean categorical text
categorical_cols = train.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    if col in test.columns:
        train[col] = train[col].astype("string").str.strip()
        test[col] = test[col].astype("string").str.strip()

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_11924\754274685.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = train.select_dtypes(include=["object"]).columns


In [22]:
for df in [train, test]:

    # Damage relative to Pikachu's maximum HP
    df["damage_pct"] = (
        df["damage_dealt"] /
        df["max_hp"].replace(0, np.nan)
    )

    # Healing relative to max HP
    df["healing_pct"] = (
        df["healing_applied"] /
        df["max_hp"].replace(0, np.nan)
    )

    # Previous HP relative to maximum HP
    df["previous_hp_pct"] = (
        df["previous_hp"] /
        df["max_hp"].replace(0, np.nan)
    )

In [23]:
for df in [train, test]:

    df["move_vs_opponent"] = (
        df["move_type"].fillna("Unknown")
        + "_vs_"
        + df["opponent_type"].fillna("Unknown")
    )

In [24]:
for df in [train, test]:

    df["effective_move_power"] = (
        df["move_power"] *
        df["type_effectiveness"]
    )

In [31]:
for df in [train, test]:

    df["effective_damage_potential"] = (
        df["effective_move_power"] *
        df["move_hit"]
    )

In [32]:
for df in [train, test]:

    df["turn_squared"] = df["turn"] ** 2

    df["turn_sqrt"] = np.sqrt(df["turn"])

In [33]:
train = train.sort_values(
    ["round", "turn"]
).reset_index(drop=True)

test = test.sort_values(
    ["round", "turn"]
).reset_index(drop=True)

In [34]:
lag_columns = [
    "damage_dealt",
    "healing_applied",
    "previous_hp",
    "type_effectiveness",
    "move_power"
]

for col in lag_columns:

    train[f"{col}_lag1"] = (
        train.groupby("round")[col].shift(1)
    )

    test[f"{col}_lag1"] = (
        test.groupby("round")[col].shift(1)
    )

In [35]:
for col in [
    "damage_dealt",
    "healing_applied",
    "previous_hp"
]:

    train[f"{col}_lag2"] = (
        train.groupby("round")[col].shift(2)
    )

    test[f"{col}_lag2"] = (
        test.groupby("round")[col].shift(2)
    )

In [36]:
train["damage_last_3"] = (
    train.groupby("round")["damage_dealt"]
    .transform(
        lambda x: x.shift(1).rolling(3).sum()
    )
)

test["damage_last_3"] = (
    test.groupby("round")["damage_dealt"]
    .transform(
        lambda x: x.shift(1).rolling(3).sum()
    )
)

In [37]:
for df in [train, test]:

    df.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

In [38]:
numeric_features = train.select_dtypes(
    include=["int64", "float64"]
).columns

correlations = (
    train[numeric_features]
    .corr()["pikachu_hp"]
    .sort_values(ascending=False)
)

print(correlations)

pikachu_hp                    1.000000
trainer_focus_score           0.961998
previous_hp                   0.443397
previous_hp_pct               0.419830
damage_dealt_lag1             0.156475
max_hp                        0.131141
pikachu_level                 0.130375
speed_stat_pikachu            0.118681
opponent_level                0.116936
healing_applied               0.111496
damage_last_3                 0.110127
sp_defense_stat               0.107362
healing_pct                   0.106723
defense_stage                 0.101842
defense_stat                  0.100808
sp_attack_stat                0.092669
attack_stat                   0.091297
type_effectiveness_lag1       0.086384
speed_stat_opponent           0.051040
move_power_lag1               0.039503
damage_dealt_lag2             0.024913
experience_points             0.021611
healing_applied_lag1          0.021361
healing_applied_lag2          0.008463
turn_sqrt                     0.007511
turn                     

In [39]:
engineered = [
    "damage_pct",
    "healing_pct",
    "previous_hp_pct",
    "effective_move_power",
    "effective_damage_potential",
    "turn_squared",
    "turn_sqrt",
    "damage_dealt_lag1",
    "damage_dealt_lag2",
    "healing_applied_lag1",
    "previous_hp_lag1",
    "damage_last_3"
]

print(
    train[engineered + ["pikachu_hp"]]
    .corr()["pikachu_hp"]
    .sort_values(ascending=False)
)


pikachu_hp                    1.000000
previous_hp_pct               0.419830
damage_dealt_lag1             0.156475
damage_last_3                 0.110127
healing_pct                   0.106723
damage_dealt_lag2             0.024913
healing_applied_lag1          0.021361
turn_sqrt                     0.007511
turn_squared                  0.006105
previous_hp_lag1              0.005295
effective_move_power         -0.323775
damage_pct                   -0.346297
effective_damage_potential   -0.375413
Name: pikachu_hp, dtype: float64


In [40]:
print(
    train[engineered + ["pikachu_hp"]]
    .corr()["pikachu_hp"]
    .sort_values(ascending=False)
)

pikachu_hp                    1.000000
previous_hp_pct               0.419830
damage_dealt_lag1             0.156475
damage_last_3                 0.110127
healing_pct                   0.106723
damage_dealt_lag2             0.024913
healing_applied_lag1          0.021361
turn_sqrt                     0.007511
turn_squared                  0.006105
previous_hp_lag1              0.005295
effective_move_power         -0.323775
damage_pct                   -0.346297
effective_damage_potential   -0.375413
Name: pikachu_hp, dtype: float64


In [47]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


# ============================================================
# 1. LOAD DATA
# ============================================================

target = "pikachu_hp"


# ============================================================
# 2. CLEAN COLUMN NAMES
# ============================================================

train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

categorical_cols = train.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    train[col] = (
        train[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

    if col in test.columns:
        test[col] = (
            test[col]
            .fillna("Unknown")
            .astype(str)
            .str.strip()
        )

# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

for df in [train, test]:

    # Damage relative to maximum HP
    df["damage_pct"] = (
        df["damage_dealt"] /
        df["max_hp"].replace(0, np.nan)
    )

    # Healing relative to maximum HP
    df["healing_pct"] = (
        df["healing_applied"] /
        df["max_hp"].replace(0, np.nan)
    )

    # Previous HP relative to maximum HP
    df["previous_hp_pct"] = (
        df["previous_hp"] /
        df["max_hp"].replace(0, np.nan)
    )

    # Move type × opponent type
    df["move_vs_opponent"] = (
        df["move_type"].fillna("Unknown")
        + "_vs_"
        + df["opponent_type"].fillna("Unknown")
    )

    # Move power adjusted for effectiveness
    df["effective_move_power"] = (
        df["move_power"] *
        df["type_effectiveness"]
    )

    # Move power × effectiveness × whether it hit
    df["effective_damage_potential"] = (
        df["effective_move_power"] *
        df["move_hit"]
    )

    # Time features
    df["turn_squared"] = df["turn"] ** 2
    df["turn_sqrt"] = np.sqrt(df["turn"])


# ============================================================
# 4. SORT BY ROUND AND TURN
# ============================================================

train = train.sort_values(
    ["round", "turn"]
).reset_index(drop=True)

test = test.sort_values(
    ["round", "turn"]
).reset_index(drop=True)


# ============================================================
# 5. LAG FEATURES
# ============================================================

lag_columns = [
    "damage_dealt",
    "healing_applied",
    "previous_hp",
    "type_effectiveness",
    "move_power"
]

for col in lag_columns:

    train[f"{col}_lag1"] = (
        train.groupby("round")[col].shift(1)
    )

    test[f"{col}_lag1"] = (
        test.groupby("round")[col].shift(1)
    )


# Two-turn lag

for col in [
    "damage_dealt",
    "healing_applied",
    "previous_hp"
]:

    train[f"{col}_lag2"] = (
        train.groupby("round")[col].shift(2)
    )

    test[f"{col}_lag2"] = (
        test.groupby("round")[col].shift(2)
    )


# ============================================================
# 6. RECENT DAMAGE
# ============================================================

train["damage_last_3"] = (
    train.groupby("round")["damage_dealt"]
    .transform(
        lambda x: x.shift(1).rolling(3).sum()
    )
)

test["damage_last_3"] = (
    test.groupby("round")["damage_dealt"]
    .transform(
        lambda x: x.shift(1).rolling(3).sum()
    )
)

# ============================================================
# 7. REMOVE INFINITE VALUES
# ============================================================

train.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

test.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)


# ============================================================
# 8. X AND y
# ============================================================

X = train.drop(columns=["pikachu_hp", "battle_turn"])
X_test = test.drop(columns=["battle_turn"])
y = train[target]



# ============================================================
# 9. IDENTIFY FEATURE TYPES
# ============================================================

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Number of numerical features:",
      len(numeric_features))

print("Number of categorical features:",
      len(categorical_features))


# ============================================================
# 10. PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),

    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


# ============================================================
# 11. MODEL
# ============================================================

model = Pipeline([
    ("preprocessor", preprocessor),

    ("regression", LinearRegression())
])


# ============================================================
# 12. TEMPORAL VALIDATION
# ============================================================

split = int(len(X) * 0.90)

X_train = X.iloc[:split]
X_valid = X.iloc[split:]

y_train = y.iloc[:split]
y_valid = y.iloc[split:]

print("\nTraining rows:", len(X_train))
print("Validation rows:", len(X_valid))


# ============================================================
# 13. TRAIN
# ============================================================

model.fit(
    X_train,
    y_train
)


# ============================================================
# 14. VALIDATION
# ============================================================

valid_predictions = model.predict(
    X_valid
)

validation_r2 = r2_score(
    y_valid,
    valid_predictions
)

print("\nDay 3 Validation R²:",
      validation_r2)


# ============================================================
# 15. TRAIN ON FULL DATA
# ============================================================

model.fit(
    X,
    y
)


# ============================================================
# 16. PREDICT TEST DATA
# ============================================================

test_predictions = model.predict(
    X_test
)


# ============================================================
# 17. CREATE SUBMISSION
# ============================================================

sample = pd.read_csv(
    "sample_submission.csv"
)

submission = pd.DataFrame({
    "battle_turn": sample["battle_turn"],
    "pikachu_hp": test_predictions
})


submission.to_csv(
    "submissionDay4.csv",
    index=False
)


# ============================================================
# 18. CHECK SUBMISSION
# ============================================================

print("\nSubmission preview:")
print(submission.head())

print("\nSubmission shape:")
print(submission.shape)

print("\nMissing values:")
print(submission.isnull().sum())

print("\nPrediction statistics:")
print(submission["pikachu_hp"].describe())

Number of numerical features: 40
Number of categorical features: 0

Training rows: 72014
Validation rows: 8002

Day 3 Validation R²: 0.9496718585555229

Submission preview:
               battle_turn  pikachu_hp
0  Round 1828 - Turn 00:00   48.313387
1  Round 1828 - Turn 01:00   48.439877
2  Round 1828 - Turn 02:00   29.861181
3  Round 1828 - Turn 03:00   26.701024
4  Round 1828 - Turn 04:00   18.511976

Submission shape:
(240, 2)

Missing values:
battle_turn    0
pikachu_hp     0
dtype: int64

Prediction statistics:
count    240.000000
mean      38.709392
std       17.756970
min        6.643468
25%       24.418561
50%       39.598180
75%       53.436706
max       69.337757
Name: pikachu_hp, dtype: float64
